# Keyword Extractor - How It Works

This notebook explains the keyword extraction module used in the wigglnet-data-engine project.

## Overview

The `keyword_extractor` module extracts meaningful keywords from text using Natural Language Processing (NLP) techniques. It leverages the NLTK (Natural Language Toolkit) library to identify the most important words in article text.

## Installation & Setup

First, let's ensure we have the necessary dependencies installed:

In [ ]:
# Required libraries
import sys
sys.path.insert(0, '../article_keywords/tools')

import nltk
from keyword_extractor import extract_keywords, extract_keywords_from_article

## How the Algorithm Works

The keyword extraction process follows these steps:

### 1. Text Preprocessing

- **Input Normalization**: Text is converted to lowercase
- **Tokenization**: Text is split into individual words using NLTK's `word_tokenize()`
- **Stopword Removal**: Common words ("the", "is", "and", etc.) are filtered out
- **Length Filtering**: Only words longer than 3 characters are kept
- **Alphabetic Filtering**: Only alphabetic tokens are retained (no numbers or punctuation)

### 2. Part-of-Speech (POS) Tagging

The algorithm uses POS tagging to identify the grammatical role of each word:

- **Nouns (NN, NNS, NNP, NNPS)**: Given **3x weight** (most important for keywords)
- **Adjectives (JJ, JJR, JJS)**: Given **2x weight** (moderately important)
- **Other parts of speech**: Given **1x weight**

This weighting ensures that content words (especially nouns) are prioritized over other word types.

### 3. Frequency Distribution

After weighting:
- A frequency distribution is calculated using NLTK's `FreqDist`
- The most common words (after weighting) are selected

### 4. Confidence Scoring

Each keyword receives:
- **Frequency**: Raw weighted count
- **Confidence**: Normalized score (0-1) representing the keyword's importance relative to other keywords

Formula: `confidence = keyword_count / total_count_of_all_keywords`

## Example: Basic Keyword Extraction

Let's see the keyword extractor in action with a sample text:

In [ ]:
# Sample article text about technology
sample_text = """
Artificial intelligence is revolutionizing the technology industry. Machine learning algorithms 
are becoming increasingly sophisticated, enabling computers to perform complex tasks that once 
required human intelligence. Deep learning neural networks have achieved remarkable breakthroughs 
in computer vision, natural language processing, and speech recognition. Companies are investing 
heavily in AI research and development to stay competitive in the rapidly evolving tech landscape. 
The applications of artificial intelligence span across healthcare, finance, transportation, and 
many other sectors. As AI technology continues to advance, it raises important questions about 
ethics, privacy, and the future of work.
"""

# Extract keywords
keywords = extract_keywords(sample_text, num_keywords=10)

# Display results
print("Extracted Keywords:\n")
for i, kw in enumerate(keywords, 1):
    print(f"{i}. {kw['keyword']:20s} | Frequency: {kw['frequency']:3d} | Confidence: {kw['confidence']:.3f}")

## Understanding the Output

Each keyword object contains three fields:

1. **`keyword`**: The extracted word
2. **`frequency`**: The weighted count (after POS weighting)
3. **`confidence`**: A normalized score (0-1) showing relative importance

Notice how domain-specific nouns like "intelligence", "learning", "technology" rank highly due to:
- High frequency in the text
- Being nouns (3x weight multiplier)
- Not being stopwords

## Example: Comparing Different Texts

Let's compare keyword extraction from texts on different topics:

In [ ]:
# Article about climate change
climate_text = """
Climate change poses significant challenges to global ecosystems and human societies. 
Rising temperatures are causing polar ice caps to melt, leading to sea level rise. 
Extreme weather events are becoming more frequent, including devastating hurricanes, 
droughts, and wildfires. Scientists emphasize the urgent need to reduce greenhouse gas 
emissions and transition to renewable energy sources. International climate agreements 
aim to limit global warming and protect the environment for future generations.
"""

# Article about cooking
cooking_text = """
Italian cuisine celebrates fresh ingredients and simple preparation techniques. 
Pasta dishes showcase regional variations, from creamy carbonara to robust marinara sauce. 
Olive oil forms the foundation of Mediterranean cooking, enhancing flavors while providing 
health benefits. Traditional Italian recipes have been passed down through generations, 
preserving culinary heritage. Fresh basil, ripe tomatoes, and quality parmesan cheese 
create authentic Italian flavors that delight food enthusiasts worldwide.
"""

# Extract keywords from both
climate_keywords = extract_keywords(climate_text, num_keywords=8)
cooking_keywords = extract_keywords(cooking_text, num_keywords=8)

print("CLIMATE CHANGE KEYWORDS:")
print("-" * 50)
for kw in climate_keywords:
    print(f"  - {kw['keyword']} (confidence: {kw['confidence']:.3f})")

print("\nCOOKING KEYWORDS:")
print("-" * 50)
for kw in cooking_keywords:
    print(f"  - {kw['keyword']} (confidence: {kw['confidence']:.3f})")

## Stopwords and Filtering

The extractor includes an extended stopword list beyond NLTK's default. This removes common but semantically empty words like:

- **Reporting verbs**: "said", "told", "according"
- **Vague adjectives**: "new", "first", "last"
- **Common verbs**: "would", "could", "may", "get", "make"
- **Generic nouns**: "people", "time", "year"

This ensures only meaningful content words appear in the results.

In [ ]:
# Example showing how stopwords affect results
text_with_common_words = """
The president said that people think the new policy would make a big difference. 
Last year, officials told reporters that the government could implement changes. 
According to experts, this year may bring important reforms.
"""

keywords_filtered = extract_keywords(text_with_common_words, num_keywords=10)

print("Keywords (after stopword filtering):")
for kw in keywords_filtered:
    print(f"  - {kw['keyword']}")

print("\nNotice that words like 'said', 'told', 'people', 'year' are filtered out!")

## Using with Article Data

The module provides helper functions for processing article dictionaries:

In [ ]:
# Example article structure (as used by web scrapers)
article = {
    'title': 'Breakthrough in Renewable Energy',
    'paragraphs': [
        'Scientists have developed a new solar panel technology that dramatically increases efficiency.',
        'The innovative panels use advanced photovoltaic materials to capture more sunlight.',
        'This breakthrough could accelerate the transition to clean energy worldwide.',
        'Researchers believe the technology will reduce solar energy costs significantly.'
    ]
}

# Extract keywords from article
article_keywords = extract_keywords_from_article(article, num_keywords=10)

print("Article Keywords:")
for kw in article_keywords:
    print(f"  - {kw['keyword']:15s} (freq: {kw['frequency']}, conf: {kw['confidence']:.3f})")

## Integration with Scrapers

The keyword extractor is used by both:

1. **`web_scraper.py`** (line 274): Processes HTML articles scraped via Selenium
2. **`rss_scraper.py`** (line 158): Processes articles from RSS feeds

Both scrapers call `add_keywords_to_articles()` which:
- Processes a batch of articles
- Extracts keywords from each article's paragraphs
- Adds a `keywords` field to each article dictionary
- Displays progress and top keywords during processing

## Key Parameters

### `num_keywords`
Controls how many keywords to extract (default: 10-15)
- **Lower values** (5-8): Only the most prominent keywords
- **Higher values** (15-20): Broader keyword coverage

### Minimum Word Length
Currently set to 4 characters (line 71 in keyword_extractor.py)
- Filters out short words like "run", "use", "get"
- Keeps more meaningful words like "running", "using", "getting"

In [ ]:
# Comparing different num_keywords values
sample = "Quantum computing represents a paradigm shift in computational technology. "\
         "Quantum computers leverage quantum mechanics principles like superposition and entanglement. "\
         "These machines can solve complex problems exponentially faster than classical computers."

print("Top 5 keywords:")
for kw in extract_keywords(sample, num_keywords=5):
    print(f"  - {kw['keyword']}")

print("\nTop 10 keywords:")
for kw in extract_keywords(sample, num_keywords=10):
    print(f"  - {kw['keyword']}")

## Algorithm Strengths

1. **Domain-Agnostic**: Works on any topic without training
2. **Fast**: No ML model training required
3. **Transparent**: Clear weighting rules based on POS tags
4. **Customizable**: Easy to adjust stopwords and weights
5. **Confidence Scores**: Provides interpretable relevance metrics

## Limitations

1. **No Context Understanding**: Doesn't understand word meanings or relationships
2. **Frequency-Based**: May miss important but infrequent terms
3. **Single-Word Only**: Doesn't extract multi-word phrases (e.g., "machine learning")
4. **Language**: Only works for English text

## Potential Improvements

- **N-gram extraction**: Capture phrases like "artificial intelligence"
- **TF-IDF weighting**: Consider word importance across multiple documents
- **Named Entity Recognition**: Specifically extract names, places, organizations
- **Semantic similarity**: Group related keywords together

## Summary

The keyword extractor uses a **rule-based NLP approach** that:

1. Tokenizes and filters text (stopwords, length, alphabetic)
2. Applies POS tagging to identify word types
3. Weights nouns 3x, adjectives 2x, others 1x
4. Calculates frequency distribution on weighted words
5. Returns top keywords with frequency and confidence scores

This approach provides reliable, interpretable keyword extraction suitable for categorizing and indexing scraped articles.